# 🎙️ Callum Offline Voice Clone for Android (Piper TTS)

Trains a custom **Piper ONNX** voice model using your Callum audio file (`callum.m4a`).
Exports `callum.onnx` and `callum.onnx.json` directly to your Google Drive to load into Android.

**Prerequisites:**
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Put your downloaded `callum.m4a` file in your **Google Drive**.

## Step 1: Mount Google Drive & Install Dependencies in Python 3.10

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# System audio dependencies
!apt-get update -qq
!apt-get install -y -qq espeak-ng ffmpeg build-essential

# Set up isolated Python 3.10 environment (Piper requires 3.10 for pre-built wheels)
!pip install -q uv
!uv venv /content/piper_env --python 3.10

# Install all required packages into Python 3.10
!/content/piper_env/bin/pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!/content/piper_env/bin/pip install -q pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize==1.1.0 onnx onnxruntime librosa cython

# Slicing tools for Step 2
!pip install -q openai-whisper pydub

print('✅ Python 3.10 environment & all libraries installed successfully!')

## Step 2: Auto-Slice & Transcribe Audio with Whisper
(If already processed, it will reuse existing slices automatically)

In [ ]:
import os
import glob

meta_path = '/content/dataset/metadata.csv'
if os.path.isfile(meta_path) and os.path.getsize(meta_path) > 500:
    print('⚡ Found existing dataset slices in /content/dataset! Skipping transcription to save time.')
else:
    import whisper
    from pydub import AudioSegment

    candidates = glob.glob('/content/drive/MyDrive/**/callum.*', recursive=True)
    if not candidates:
        candidates = glob.glob('/content/drive/MyDrive/callum.*')
    if not candidates:
        raise FileNotFoundError('Could not find callum audio file in Google Drive! Please upload callum.m4a to Google Drive.')

    audio_path = candidates[0]
    print(f'Found audio file: {audio_path}')

    # 1. Load Whisper ASR model
    print('Loading Whisper (base.en)...')
    asr_model = whisper.load_model('base.en')

    # 2. Transcribe and extract timestamps
    print('Transcribing audio and extracting timestamps (~3-5 mins)...')
    result = asr_model.transcribe(audio_path, language='en')

    # 3. Convert source audio to 22050Hz Mono 16-bit
    print('Processing audio slices to 22050 Hz Mono...')
    sound = AudioSegment.from_file(audio_path).set_frame_rate(22050).set_channels(1).set_sample_width(2)
    os.makedirs('/content/dataset/wav', exist_ok=True)

    # 4. Export sliced WAVs and metadata.csv
    count = 0
    with open(meta_path, 'w', encoding='utf-8') as f:
        for seg in result['segments']:
            start_ms = int(seg['start'] * 1000)
            end_ms = int(seg['end'] * 1000)
            duration = end_ms - start_ms
            text = seg['text'].strip()
            if 1500 <= duration <= 10000 and len(text) > 3:
                chunk = sound[start_ms:end_ms]
                file_id = f'{count:05d}'
                chunk.export(f'/content/dataset/wav/{file_id}.wav', format='wav')
                f.write(f'{file_id}|{text}\n')
                count += 1

    print(f'✅ Dataset ready! Created {count} slices in /content/dataset.')

## Step 3: Install Piper Training Engine & Download Base Checkpoint

In [ ]:
import os, urllib.request

# 1. Clone repository
!rm -rf /content/piper
!git clone -q https://github.com/rhasspy/piper.git /content/piper

# 2. Build C++ alignment module
!cd /content/piper/src/python/piper_train/vits/monotonic_align && /content/piper_env/bin/cythonize -i core.pyx && mkdir -p monotonic_align && cp core*.so monotonic_align/

# 3. Install piper_train package into Python 3.10
!cd /content/piper/src/python && /content/piper_env/bin/pip install -q --no-deps -e .

# 4. Download base checkpoint (en_US-lessac-medium)
os.makedirs('/content/base_model', exist_ok=True)
base_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
print('Downloading base model checkpoint...')
urllib.request.urlretrieve(base_url, '/content/base_model/base.ckpt')
print('✅ Piper training engine and base checkpoint ready!')

## Step 4: Preprocess Dataset into Phonemes

In [ ]:
!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train.preprocess \
  --language en-us \
  --input-dir /content/dataset \
  --output-dir /content/preprocessed \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

print('✅ Dataset preprocessed into phonemes successfully!')

## Step 5: Fine-Tune Callum Voice on GPU
Trains for ~2,000 steps (~25-35 mins).

In [ ]:
!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train \
  --dataset-dir /content/preprocessed \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --checkpoint-epochs 1 \
  --max_epochs 2205 \
  --resume_from_checkpoint /content/base_model/base.ckpt

## Step 6: Export to ONNX and Save to Google Drive

In [ ]:
import glob, os, shutil
drive_output_dir = '/content/drive/MyDrive/Callum_Voice'
os.makedirs(drive_output_dir, exist_ok=True)

checkpoints = glob.glob('/content/preprocessed/lightning_logs/**/checkpoints/*.ckpt', recursive=True)
if not checkpoints:
    raise FileNotFoundError('No checkpoint found!')

latest_ckpt = max(checkpoints, key=os.path.getmtime)
print(f'Latest checkpoint: {latest_ckpt}')

onnx_output = os.path.join(drive_output_dir, 'callum.onnx')
!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train.export_onnx \
  --checkpoint "{latest_ckpt}" \
  --output-file "{onnx_output}"

base_config = '/content/preprocessed/config.json'
json_output = os.path.join(drive_output_dir, 'callum.onnx.json')
shutil.copy(base_config, json_output)

print('\n🎉 SUCCESS! Exported to Google Drive in Callum_Voice folder:')
print(f'1. {onnx_output}')
print(f'2. {json_output}')